**Import Data**

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_csv('paket_a_transaksi_pos_2024_2025.csv')
df.head(10)

,id_transaksi,tanggal,kode_gerai,kode_kasir,kode_produk,jumlah,harga_satuan,diskon_persen,status
0,TRX0241065,2025-06-02,G09,K011,RMT1402,4,13000,0,SELESAI
1,TRX0240510,2024-12-28,G09,K015,SGR1205,1,21000,0,SELESAI
2,TRX0244483,2024-09-04,G11,K029,RMT1407,3,"7500,00",0,SELESAI
3,TRX0242532,2025-07-30,G05,K001,MKN1103,2,6500,0,SELESAI
4,TRX0243509,26/03/2025,G05,K016,RMT1407,1,6000,0,SELESAI
5,TRX0240658,2025-07-18,G01,K008,RMT1408,12,45500,10,SELESAI
6,TRX0244581,2025-09-24,G10,K021,MIN1004,1,18000,0,SELESAI
7,TRX0241251,2024-10-09,G09,K035,SGR1204,4,47500,10,SELESAI
8,TRX0244274,2024-03-20,G07,K011,BYI1508,3,"11000,00",0,SELESAI
9,TRX0243163,2024-03-22,G08,K028,PRW1304,2,6500,0,BATAL


**Memeriksa tipe data, nilai kosong, duplikasi, dan nilai kategorikal**

In [ ]:
#struktur data: tipe kolom dan jumlah non-null
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5030 entries, 0 to 5029
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   id_transaksi   5030 non-null   object
 1   tanggal        5030 non-null   object
 2   kode_gerai     5030 non-null   object
 3   kode_kasir     4769 non-null   object
 4   kode_produk    5030 non-null   object
 5   jumlah         5030 non-null   int64 
 6   harga_satuan   5030 non-null   object
 7   diskon_persen  5030 non-null   int64 
 8   status         5030 non-null   object
dtypes: int64(2), object(7)
memory usage: 353.8+ KB


In [ ]:
#jumlah nilai kosong, baris duplikat, dan sebaran status transaksi
print('Nilai kosong per kolom:')
print(df.isna().sum(), '\n')
print('Baris duplikat persis      :', df.duplicated().sum())
print('Jumlah ID transaksi duplikat:', df['id_transaksi'].duplicated().sum(), '\n')
print('Sebaran status:')
print(df['status'].value_counts())

Nilai kosong per kolom:
id_transaksi       0
tanggal            0
kode_gerai         0
kode_kasir       261
kode_produk        0
jumlah             0
harga_satuan       0
diskon_persen      0
status             0
dtype: int64 

Baris duplikat persis      : 30
Jumlah ID transaksi duplikat: 30 

Sebaran status:
status
SELESAI    4749
BATAL       191
RETUR        90
Name: count, dtype: int64


**Pembersihan**

In [ ]:
# 1.hapus duplikat persis, simpan baris pertama saja
jumlah_awal = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print(f'Baris dihapus karena duplikat: {jumlah_awal - len(df)} (sisa {len(df)} baris)')

# 2.merapihkan spasi pada kolom teks kode
for kolom in ['id_transaksi', 'kode_gerai', 'kode_produk', 'status']:
    df[kolom] = df[kolom].str.strip()

# 3.seragamkan tanggal: coba tiap format satu per satu, isi yang gagal dengan format berikutnya
tgl_iso   = pd.to_datetime(df['tanggal'], format='%Y-%m-%d', errors='coerce')
tgl_slash = pd.to_datetime(df['tanggal'], format='%d/%m/%Y', errors='coerce')
tgl_dash  = pd.to_datetime(df['tanggal'], format='%d-%m-%Y', errors='coerce')
df['tanggal'] = tgl_iso.fillna(tgl_slash).fillna(tgl_dash)

print('Tanggal gagal terbaca (NaT):', df['tanggal'].isna().sum())
print('Rentang tanggal:', df['tanggal'].min().date(), 's.d.', df['tanggal'].max().date())

Baris dihapus karena duplikat: 30 (sisa 5000 baris)
Tanggal gagal terbaca (NaT): 0
Rentang tanggal: 2024-01-01 s.d. 2025-12-31


#**TASK 1 - Indexing & Selection**

**1.1 Pembersihan Format Harga**

In [ ]:
#nilai sebelum dibersihkan
print(df.loc[df['harga_satuan'].str.contains('[.,]'), 'harga_satuan'].head(8).tolist())

harga = df['harga_satuan'].astype(str).str.strip()

#menghapus titik pemisah ribuan
harga = harga.str.replace(r'\.(?=\d{3}(?:\.|$))', '', regex=True)
#mengganti koma desimal menjadi titik
harga = harga.str.replace(',', '.', regex=False)

#konversi ke float; nilai yang tidak valid menjadi NaN
df['harga_satuan'] = pd.to_numeric(harga, errors='coerce')

print('\nTipe data      :', df['harga_satuan'].dtype)
print('Nilai NaN      :', df['harga_satuan'].isna().sum())
print('Min / Max harga:', df['harga_satuan'].min(), '/', df['harga_satuan'].max())

['7500,00', '11000,00', '13.000', '27000,00', '13500,00', '28.000', '43500,00', '16500,00']

Tipe data      : float64
Nilai NaN      : 0
Min / Max harga: 4000.0 / 54000.0


**1.2 Penanganan Kuantitas Negatif**

In [ ]:
print('Jumlah negatif sebelum:', (df['jumlah'] < 0).sum())

#mengubah nilai jumlah < 0 menjadi positifnya
df.loc[df['jumlah'] < 0, 'jumlah'] = df.loc[df['jumlah'] < 0, 'jumlah'].abs()

print('Jumlah negatif sesudah:', (df['jumlah'] < 0).sum())

Jumlah negatif sebelum: 40
Jumlah negatif sesudah: 0


**Imputasi Nilai Kosong Kasir**

In [ ]:
print('NaN kode_kasir sebelum:', df['kode_kasir'].isna().sum())

df.loc[df['kode_kasir'].isna(), 'kode_kasir'] = 'K_UNKNOWN'

print('NaN kode_kasir sesudah:', df['kode_kasir'].isna().sum())
df['kode_kasir'].value_counts().head()

NaN kode_kasir sebelum: 260
NaN kode_kasir sesudah: 0


,count
kode_kasir,
K_UNKNOWN,260
K021,142
K008,138
K012,134
K025,134


**1.4 Seleksi Posisi Slicing**

In [ ]:
#mencari posisi indeks angka kolom yang dibutuhkan
posisi_kolom = df.columns.get_indexer(['id_transaksi', 'tanggal', 'kode_gerai', 'jumlah'])

#5 baris pertama
subset_iloc = df.iloc[:5, posisi_kolom]
subset_iloc

,id_transaksi,tanggal,kode_gerai,jumlah
0,TRX0241065,2025-06-02,G09,4
1,TRX0240510,2024-12-28,G09,1
2,TRX0244483,2024-09-04,G11,3
3,TRX0242532,2025-07-30,G05,2
4,TRX0243509,2025-03-26,G05,1


**Kesimpulan Task 1:**
Pada Task 1, kolom harga_satuan yang awalnya masih berbentuk teks berhasil diubah menjadi tipe data angka. Dalam prosesnya, ditemukan dua format penulisan harga, yaitu menggunakan koma sebagai tanda desimal (7500,00) dan titik sebagai pemisah ribuan (13.500). Kedua format tersebut disesuaikan agar nilai harga dapat terbaca dengan benar. Selain itu, terdapat 40 data jumlah yang bernilai negatif dan diubah menjadi positif menggunakan .loc[] dan .abs(). Sebanyak 260 data kasir yang kosong juga diisi dengan K_UNKNOWN. Terakhir, penggunaan .iloc[] dilakukan untuk menampilkan lima baris pertama berdasarkan posisi kolom yang diminta.

#**TASK 2 — Operasi Aritmatika (Feature Engineering)**

**2.1 Perhitungan Omzet Kotor**

In [ ]:
df['Total_Kotor'] = df['harga_satuan'] * df['jumlah']
df[['id_transaksi', 'harga_satuan', 'jumlah', 'Total_Kotor']].head()

,id_transaksi,harga_satuan,jumlah,Total_Kotor
0,TRX0241065,13000.0,4,52000.0
1,TRX0240510,21000.0,1,21000.0
2,TRX0244483,7500.0,3,22500.0
3,TRX0242532,6500.0,2,13000.0
4,TRX0243509,6000.0,1,6000.0


**2.2 Perhitungan Diskon**

In [ ]:
df['Nominal_Diskon'] = df['Total_Kotor'] * (df['diskon_persen'] / 100)
df[['id_transaksi', 'Total_Kotor', 'diskon_persen', 'Nominal_Diskon']].head()

,id_transaksi,Total_Kotor,diskon_persen,Nominal_Diskon
0,TRX0241065,52000.0,0,0.0
1,TRX0240510,21000.0,0,0.0
2,TRX0244483,22500.0,0,0.0
3,TRX0242532,13000.0,0,0.0
4,TRX0243509,6000.0,0,0.0


**2.3 Perhitungan Omzet Bersih & Pajak**

In [ ]:
df['Total_Bersih'] = (df['Total_Kotor'] - df['Nominal_Diskon']) * 1.11
df

,id_transaksi,tanggal,kode_gerai,kode_kasir,kode_produk,jumlah,harga_satuan,diskon_persen,status,Total_Kotor,Nominal_Diskon,Total_Bersih
0,TRX0241065,2025-06-02,G09,K011,RMT1402,4,13000.0,0,SELESAI,52000.0,0.0,57720.0
1,TRX0240510,2024-12-28,G09,K015,SGR1205,1,21000.0,0,SELESAI,21000.0,0.0,23310.0
2,TRX0244483,2024-09-04,G11,K029,RMT1407,3,7500.0,0,SELESAI,22500.0,0.0,24975.0
3,TRX0242532,2025-07-30,G05,K001,MKN1103,2,6500.0,0,SELESAI,13000.0,0.0,14430.0
4,TRX0243509,2025-03-26,G05,K016,RMT1407,1,6000.0,0,SELESAI,6000.0,0.0,6660.0
...,...,...,...,...,...,...,...,...,...,...,...,...
4995,TRX0241722,2024-01-17,G01,K024,MIN1008,2,5500.0,0,SELESAI,11000.0,0.0,12210.0
4996,TRX0243698,2025-02-28,G01,K021,PRW1302,4,20500.0,0,SELESAI,82000.0,0.0,91020.0
4997,TRX0241377,2024-12-06,G08,K008,RMT1407,1,7000.0,0,SELESAI,7000.0,0.0,7770.0
4998,TRX0241303,2025-10-01,G05,K008,MIN1003,1,29000.0,15,SELESAI,29000.0,4350.0,27361.5


**Kesimpulan Task 2:** Pada Task 2, dibuat tiga kolom tambahan menggunakan operasi vektor, yaitu Total_Kotor, Nominal_Diskon, dan Total_Bersih. Kolom Total_Kotor digunakan untuk menghitung total harga sebelum diskon. Selanjutnya, Nominal_Diskon digunakan untuk menghitung besarnya potongan harga dengan membagi nilai diskon sebesar 100 karena datanya menggunakan skala 0–100. Setelah itu, Total_Bersih dihitung berdasarkan total setelah diskon dengan tambahan PPN sebesar 11%.



#**TASK 3 — Filtering (Penyaringan Data)**

**3.1 Filter Multi-Kondisi**

In [ ]:
filter_multi = df[
    (df['status'] == 'SELESAI') &
    (df['kode_gerai'].isin(['G05', 'G09'])) &
    (df['Total_Bersih'] > 100000)
]

print('Jumlah baris yang memenuhi kriteria:', len(filter_multi))
filter_multi

Jumlah baris yang memenuhi kriteria: 139


,id_transaksi,tanggal,kode_gerai,kode_kasir,kode_produk,jumlah,harga_satuan,diskon_persen,status,Total_Kotor,Nominal_Diskon,Total_Bersih
7,TRX0241251,2024-10-09,G09,K035,SGR1204,4,47500.0,10,SELESAI,190000.0,19000.0,189810.0
24,TRX0244253,2024-07-18,G09,K026,BYI1502,4,30500.0,0,SELESAI,122000.0,0.0,135420.0
25,TRX0244999,2024-08-09,G05,K015,MIN1003,4,28500.0,0,SELESAI,114000.0,0.0,126540.0
90,TRX0240898,2025-12-09,G05,K010,BYI1509,3,46500.0,0,SELESAI,139500.0,0.0,154845.0
206,TRX0241868,2025-07-01,G05,K009,SGR1203,6,20500.0,0,SELESAI,123000.0,0.0,136530.0
...,...,...,...,...,...,...,...,...,...,...,...,...
4849,TRX0243898,2024-04-23,G05,K008,SGR1204,12,51000.0,0,SELESAI,612000.0,0.0,679320.0
4883,TRX0241966,2024-04-22,G09,K024,SGR1206,3,42000.0,0,SELESAI,126000.0,0.0,139860.0
4897,TRX0240377,2025-06-11,G09,K028,XXX9969,12,51500.0,15,SELESAI,618000.0,92700.0,583083.0
4935,TRX0243219,2025-11-17,G09,K003,SGR1204,6,43500.0,0,SELESAI,261000.0,0.0,289710.0


**3.2 Filter dengan Query**

In [ ]:
filter_g10 = df.query('kode_gerai == "G10" and jumlah >= 5')

print('Jumlah baris:', len(filter_g10))
filter_g10

Jumlah baris: 39


,id_transaksi,tanggal,kode_gerai,kode_kasir,kode_produk,jumlah,harga_satuan,diskon_persen,status,Total_Kotor,Nominal_Diskon,Total_Bersih
83,TRX0243880,2024-08-04,G10,K002,MKN1101,12,45500.0,0,SELESAI,546000.0,0.0,606060.0
166,TRX0241592,2024-05-11,G10,K033,SGR1202,6,13000.0,0,SELESAI,78000.0,0.0,86580.0
351,TRX0242686,2024-02-03,G10,K016,BYI1502,12,26000.0,0,SELESAI,312000.0,0.0,346320.0
555,TRX0241396,2025-12-16,G10,K009,RMT1406,12,5500.0,5,SELESAI,66000.0,3300.0,69597.0
878,TRX0243559,2025-09-30,G10,K016,SGR1206,6,47000.0,0,SELESAI,282000.0,0.0,313020.0
963,TRX0244939,2025-07-08,G10,K009,SGR1200,6,52000.0,10,SELESAI,312000.0,31200.0,311688.0
988,TRX0243076,2025-02-18,G10,K003,MKN1100,6,18000.0,0,SELESAI,108000.0,0.0,119880.0
1022,TRX0242437,2025-09-18,G10,K030,MKN1104,6,6000.0,0,SELESAI,36000.0,0.0,39960.0
1257,TRX0240537,2025-05-09,G10,K027,BYI1509,6,54000.0,0,SELESAI,324000.0,0.0,359640.0
1293,TRX0240698,2024-01-24,G10,K022,MKN1109,6,29000.0,0,SELESAI,174000.0,0.0,193140.0


**Kesimpulan Task 3:** Pada Task 3, dilakukan penyaringan data menggunakan beberapa kondisi sekaligus. Filter pertama digunakan untuk mengambil transaksi dengan status SELESAI, berasal dari gerai G05 atau G09, serta memiliki Total_Bersih di atas Rp100.000. Dari proses tersebut diperoleh 139 transaksi. Selanjutnya, penggunaan .query() dilakukan untuk mencari transaksi dari gerai G10 dengan jumlah pembelian minimal 5 unit dan menghasilkan 39 transaksi.

#**TASK 4 — Sorting (Pengurutan Data)**

**4.1 Multi-Column Sorting**

In [ ]:
df_sorted = df.sort_values(by=['kode_gerai', 'Total_Bersih'], ascending=[True, False])
df_sorted

,id_transaksi,tanggal,kode_gerai,kode_kasir,kode_produk,jumlah,harga_satuan,diskon_persen,status,Total_Kotor,Nominal_Diskon,Total_Bersih
244,TRX0243257,2024-11-24,G01,K033,MKN1102,1456,32500.0,20,SELESAI,47320000.0,9464000.0,42020160.0
2946,TRX0241367,2025-05-15,G01,K006,RMT1408,12,50500.0,0,SELESAI,606000.0,0.0,672660.0
3573,TRX0240608,2025-07-25,G01,K006,MKN1101,12,47000.0,0,SELESAI,564000.0,0.0,626040.0
2927,TRX0240631,2024-12-24,G01,K013,RMT1408,12,42000.0,0,SELESAI,504000.0,0.0,559440.0
5,TRX0240658,2025-07-18,G01,K008,RMT1408,12,45500.0,10,SELESAI,546000.0,54600.0,545454.0
...,...,...,...,...,...,...,...,...,...,...,...,...
3708,TRX0242116,2025-10-31,G12,K024,MIN1008,1,4500.0,0,SELESAI,4500.0,0.0,4995.0
1482,TRX0241390,2024-12-29,G12,K008,PRW1300,1,5500.0,20,SELESAI,5500.0,1100.0,4884.0
2286,TRX0244124,2024-01-26,G12,K035,MIN1008,1,5500.0,20,SELESAI,5500.0,1100.0,4884.0
3833,TRX0242420,2024-01-29,G12,K019,RMT1406,1,4000.0,0,SELESAI,4000.0,0.0,4440.0


**4.2 Top-N Records**

In [ ]:
top_5_transaksi = df.nlargest(5, 'Total_Bersih')
top_5_transaksi

,id_transaksi,tanggal,kode_gerai,kode_kasir,kode_produk,jumlah,harga_satuan,diskon_persen,status,Total_Kotor,Nominal_Diskon,Total_Bersih
993,TRX0241002,2024-03-18,G11,K038,MKN1101,2093,48000.0,10,SELESAI,100464000.0,10046400.0,100363536.0
4956,TRX0244221,2024-12-09,G09,K005,BYI1500,1301,51500.0,10,SELESAI,67001500.0,6700150.0,66934498.5
466,TRX0243696,2024-02-27,G06,K_UNKNOWN,SGR1201,892,54000.0,0,SELESAI,48168000.0,0.0,53466480.0
3379,TRX0241029,2025-05-12,G07,K028,SGR1205,1983,23000.0,10,SELESAI,45609000.0,4560900.0,45563391.0
244,TRX0243257,2024-11-24,G01,K033,MKN1102,1456,32500.0,20,SELESAI,47320000.0,9464000.0,42020160.0


**Kesimpulan Task 4:** Pada Task 4, data diurutkan berdasarkan kode_gerai dari A sampai Z, kemudian berdasarkan Total_Bersih dari nilai terbesar ke terkecil. Selain itu, digunakan .nlargest() untuk mengambil lima transaksi dengan nilai tertinggi. Dengan cara ini, transaksi yang memiliki nilai penjualan paling besar dapat diketahui dengan lebih mudah.

#**TASK 5 — Grouping & Agregasi (Pelaporan Kinerja Gerai)**

**5.1 Laporan Agregasi Per Gerai**

In [ ]:
df_selesai = df[df['status'] == 'SELESAI']

laporan_gerai = df_selesai.groupby('kode_gerai', as_index=False).agg(
    Jumlah_Transaksi=('id_transaksi', 'count'),
    Total_Unit_Terjual=('jumlah', 'sum'),
    Total_Omzet_Bersih=('Total_Bersih', 'sum'),
    Rata_Rata_Transaksi=('Total_Bersih', 'mean')
).sort_values(by='Total_Omzet_Bersih', ascending=False).round(2)

laporan_gerai

,kode_gerai,Jumlah_Transaksi,Total_Unit_Terjual,Total_Omzet_Bersih,Rata_Rata_Transaksi
10,G11,395,3162,1.253962e+08,317458.67
8,G09,376,3018,1.008712e+08,268274.45
5,G06,381,3099,9.110042e+07,239108.71
6,G07,380,4266,8.033608e+07,211410.75
0,G01,445,2626,7.058754e+07,158623.68
2,G03,370,3268,5.988800e+07,161859.45
9,G10,400,1741,5.893936e+07,147348.41
7,G08,406,1696,3.551953e+07,87486.52
3,G04,397,1038,2.457493e+07,61901.58
11,G12,401,1001,2.442228e+07,60903.43


**5.2 Transformasi Kelompok (Window Function)**

In [ ]:
#total omzet bersih per gerai, diulang pada setiap baris gerai tersebut
total_per_gerai = df.groupby('kode_gerai')['Total_Bersih'].transform('sum')

#persentase kontribusi tiap transaksi terhadap total gerainya
df['Kontribusi_Gerai_Persen'] = df['Total_Bersih'] / total_per_gerai * 100

#validasi: total persentase tiap gerai harus 100
print(df.groupby('kode_gerai')['Kontribusi_Gerai_Persen'].sum().round(6).unique())
df

[100.]


,id_transaksi,tanggal,kode_gerai,kode_kasir,kode_produk,jumlah,harga_satuan,diskon_persen,status,Total_Kotor,Nominal_Diskon,Total_Bersih,Kontribusi_Gerai_Persen
0,TRX0241065,2025-06-02,G09,K011,RMT1402,4,13000.0,0,SELESAI,52000.0,0.0,57720.0,0.056450
1,TRX0240510,2024-12-28,G09,K015,SGR1205,1,21000.0,0,SELESAI,21000.0,0.0,23310.0,0.022797
2,TRX0244483,2024-09-04,G11,K029,RMT1407,3,7500.0,0,SELESAI,22500.0,0.0,24975.0,0.018359
3,TRX0242532,2025-07-30,G05,K001,MKN1103,2,6500.0,0,SELESAI,13000.0,0.0,14430.0,0.060075
4,TRX0243509,2025-03-26,G05,K016,RMT1407,1,6000.0,0,SELESAI,6000.0,0.0,6660.0,0.027727
...,...,...,...,...,...,...,...,...,...,...,...,...,...
4995,TRX0241722,2024-01-17,G01,K024,MIN1008,2,5500.0,0,SELESAI,11000.0,0.0,12210.0,0.016914
4996,TRX0243698,2025-02-28,G01,K021,PRW1302,4,20500.0,0,SELESAI,82000.0,0.0,91020.0,0.126088
4997,TRX0241377,2024-12-06,G08,K008,RMT1407,1,7000.0,0,SELESAI,7000.0,0.0,7770.0,0.021108
4998,TRX0241303,2025-10-01,G05,K008,MIN1003,1,29000.0,15,SELESAI,29000.0,4350.0,27361.5,0.113912


**Kesimpulan Task 5:** Pada Task 5, dilakukan perhitungan laporan kinerja masing-masing gerai dengan hanya menggunakan transaksi yang berstatus SELESAI. Berdasarkan hasil pengolahan, gerai G11 memperoleh omzet bersih tertinggi, yaitu sekitar Rp125,4 juta, kemudian diikuti oleh G09 dan G06. Sementara itu, G01 memiliki jumlah transaksi terbanyak, yaitu 445 transaksi, tetapi berada di peringkat kelima berdasarkan omzet. Berbeda dengan itu, G07 menjadi gerai dengan jumlah unit terjual paling banyak, yaitu 4.266 unit.

Selain itu, kolom Kontribusi_Gerai_Persen dibuat menggunakan .groupby().transform() untuk menghitung persentase kontribusi masing-masing gerai. Hasil validasi menunjukkan bahwa total persentase kontribusi setiap gerai mencapai 100%.

#**Outlier pada jumlah**

In [ ]:
outlier_jumlah = df[df['jumlah'] > 100].sort_values('jumlah', ascending=False)

porsi = outlier_jumlah['Total_Bersih'].sum() / df['Total_Bersih'].sum() * 100
print(f'{len(outlier_jumlah)} baris outlier, menyumbang {porsi:.1f}% dari seluruh Total_Bersih')
outlier_jumlah[['id_transaksi', 'kode_gerai', 'kode_produk', 'jumlah', 'status', 'Total_Bersih']]

12 baris outlier, menyumbang 58.4% dari seluruh Total_Bersih


,id_transaksi,kode_gerai,kode_produk,jumlah,status,Total_Bersih
87,TRX0243886,G03,PRW1306,2375,SELESAI,3.822562e+07
993,TRX0241002,G11,MKN1101,2093,SELESAI,1.003635e+08
3379,TRX0241029,G07,SGR1205,1983,SELESAI,4.556339e+07
244,TRX0243257,G01,MKN1102,1456,SELESAI,4.202016e+07
2409,TRX0241473,G06,PRW1308,1310,SELESAI,1.599510e+07
4956,TRX0244221,G09,BYI1500,1301,SELESAI,6.693450e+07
925,TRX0240632,G07,RMT1403,1280,SELESAI,9.235200e+06
466,TRX0243696,G06,SGR1201,892,SELESAI,5.346648e+07
1034,TRX0241068,G09,SGR1202,759,SELESAI,1.080493e+07
3209,TRX0242632,G10,SGR1204,746,SELESAI,3.436449e+07


cell ini menyaring baris dengan jumlah di atas 100 unit. ada 12 baris (11 SELESAI dan 1 BATAL) yang menyumbang sekitar 58% dari seluruh omzet bersih. Nilainya jauh di luar pola normal (kebanyakan transaksi 1 sampai 12 unit)